In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Load the dataset
diabetes = load_diabetes()

X = diabetes.data
y = diabetes.target

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (442, 10)
y shape: (442,)


In [6]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [7]:
# Create the random forest model
rf = RandomForestRegressor(n_estimators=100, random_state = 42)

In [8]:
# Train the model
rf.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [9]:
# Make a normal Random Forest prediction
y_pred = rf.predict(X_test)

In [14]:
print(y_pred[:10])

[144.   171.58 150.72 253.71 107.19 123.54 239.09 219.01 153.44 184.59]


## For regression:

```
Tree 1 → 145
Tree 2 → 152
Tree 3 → 148
Tree 4 → 155
Tree 5 → 149

       ↓

Average

       ↓

Final ≈ 149.8 ```

In [15]:
# Manually inspect the individual trees
sample = X_test[0].reshape(1, -1)

In [21]:
individual_predictions = []

for tree in rf.estimators_:
    prediction = int(tree.predict(sample)[0])
    individual_predictions.append(prediction)

print(individual_predictions)

[196, 206, 142, 67, 115, 131, 230, 164, 200, 142, 142, 67, 109, 134, 67, 115, 170, 170, 121, 131, 196, 283, 104, 170, 115, 116, 131, 170, 131, 115, 95, 142, 109, 230, 241, 120, 131, 214, 217, 69, 142, 109, 115, 134, 230, 196, 262, 131, 214, 134, 139, 116, 138, 209, 109, 115, 115, 128, 121, 196, 235, 134, 134, 196, 131, 95, 134, 128, 115, 134, 95, 190, 116, 109, 67, 67, 116, 229, 124, 128, 198, 131, 196, 196, 131, 196, 143, 143, 120, 115, 69, 115, 115, 134, 142, 95, 131, 74, 170, 138]


In [22]:
# Now, calculate the mean
manual_prediction = np.mean(individual_predictions)

print("Manual prediction:", manual_prediction)

Manual prediction: 144.0


In [23]:
# evaluate the whole Random Forest
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 44.053033707865175
MSE : 2952.0105887640448
RMSE: 54.332408273184846
R²  : 0.4428225673999313


In [24]:
# Implementing Hyperparameter tuning for better results
from sklearn.model_selection import GridSearchCV

In [25]:
# Defining the parameters
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

In [26]:
# Create Grid Search CV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

In [27]:
# Train the model
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 5, 10, 20],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='neg_mean_squared_error')

In [28]:
# Find the best parameters
print("Best parameters:")
print(grid_search.best_params_)

Best parameters:
{'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 200}


In [29]:
# Get the best model
best_rf = grid_search.best_estimator_

In [30]:
# Predict using the tuned model
y_pred_tuned = best_rf.predict(X_test)

In [31]:
# Evaluate the tuned model
mae = mean_absolute_error(y_test, y_pred_tuned)
mse = mean_squared_error(y_test, y_pred_tuned)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_tuned)

print("Tuned Random Forest")
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

Tuned Random Forest
MAE : 43.449780317784
MSE : 2876.511883177569
RMSE: 53.633123004143336
R²  : 0.45707257554807734


In [32]:
# Comparing the original vs tuned model
original_pred = rf.predict(X_test)

original_mae = mean_absolute_error(y_test, original_pred)
original_mse = mean_squared_error(y_test, original_pred)
original_rmse = np.sqrt(original_mse)
original_r2 = r2_score(y_test, original_pred)


tuned_mae = mean_absolute_error(y_test, y_pred_tuned)
tuned_mse = mean_squared_error(y_test, y_pred_tuned)
tuned_rmse = np.sqrt(tuned_mse)
tuned_r2 = r2_score(y_test, y_pred_tuned)

In [33]:
results = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²"],
    "Original Random Forest": [
        original_mae,
        original_mse,
        original_rmse,
        original_r2
    ],
    "Tuned Random Forest": [
        tuned_mae,
        tuned_mse,
        tuned_rmse,
        tuned_r2
    ]
})

print(results)

  Metric  Original Random Forest  Tuned Random Forest
0    MAE               44.053034            43.449780
1    MSE             2952.010589          2876.511883
2   RMSE               54.332408            53.633123
3     R²                0.442823             0.457073


# The final notebook structure
```
22. Random Forest Regressor
│
├── 1. Imports
│
├── 2. Load Diabetes Dataset
│
├── 3. Train/Test Split
│
├── 4. Create Random Forest
│
├── 5. Train Model
│
├── 6. Make Predictions
│
├── 7. Individual Tree Predictions
│
├── 8. Manual Average
│
├── 9. Model Evaluation
│
│
└── 10. Hyperparameter Tuning
     │
     ├── Define parameter grid
     ├── Create GridSearchCV
     ├── Fit GridSearchCV
     ├── Find best parameters
     ├── Get best model
     ├── Predict
     ├── Evaluate
     └── Compare Original vs Tuned